# GPT 

In [ ]:
# ============================================================
# FINAL PIPELINE:
# Time-aware CV + modest hyperparameter search
# Focused on:
#   1) xgb_full_propagation
#   2) lstm_context_full
# ============================================================

from pathlib import Path
import sys
import importlib
import warnings
warnings.filterwarnings("ignore")

import os
import time
import gc
import multiprocessing
from copy import deepcopy
from itertools import product

N_CORES = multiprocessing.cpu_count()
print(f"[SYSTEM] Detected {N_CORES} CPU cores")

# Thread settings
os.environ["OMP_NUM_THREADS"] = str(N_CORES)
os.environ["MKL_NUM_THREADS"] = str(N_CORES)
os.environ["OPENBLAS_NUM_THREADS"] = str(N_CORES)
os.environ["NUMEXPR_NUM_THREADS"] = str(N_CORES)
os.environ["TF_NUM_INTEROP_THREADS"] = str(N_CORES)
os.environ["TF_NUM_INTRAOP_THREADS"] = str(N_CORES)

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    roc_curve,
)
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier, XGBRegressor

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print(f"[TF] inter_op threads: {tf.config.threading.get_inter_op_parallelism_threads()}")
print(f"[TF] intra_op threads: {tf.config.threading.get_intra_op_parallelism_threads()}")

GLOBAL_START = time.perf_counter()

def timer_log(label, start_time):
    elapsed = time.perf_counter() - start_time
    print(f"[TIMER] {label}: {elapsed:.2f}s")


# ============================================================
# DEVICE SETUP
# ============================================================

# This assumes you have a GPU and the correct driver setup. 

USE_GPU = False  # safer if you are currently CPU-only

try:
    if USE_GPU:
        gpus = tf.config.list_physical_devices("GPU")
        if gpus:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"[TF] GPU mode enabled. Found {len(gpus)} GPU(s): {[g.name for g in gpus]}")
        else:
            print("[TF] WARNING: USE_GPU=True but no GPU found — falling back to CPU.")
    else:
        tf.config.set_visible_devices([], "GPU")
        print("[TF] CPU-only mode enabled.")
except RuntimeError as e:
    print(f"[TF WARNING] Device config must be set before TF runtime init: {e}")


# ============================================================
# 1. LOAD DATA
# ============================================================

t0 = time.perf_counter()

pl.Config.set_tbl_rows(1000)
pl.Config.set_tbl_cols(100)
pl.Config.set_tbl_width_chars(200)

current = Path.cwd()
while current.name != "shared-notebooks":
    if current.parent == current:
        raise RuntimeError("Could not locate shared-notebooks directory")
    current = current.parent

utils_path = current / "common_utils" / "python"
if str(utils_path) not in sys.path:
    sys.path.append(str(utils_path))

import load_flight_data
importlib.reload(load_flight_data)

lf = load_flight_data.load_flight_data(file_name="flights_canonical_2019.parquet")

timer_log("Load data", t0)


# ============================================================
# 2. FEATURE ENGINEERING / MODELING TABLE
# ============================================================

t0 = time.perf_counter()

RAW_FEATURES = [
    "flight_id",
    "tail_number",
    "reporting_airline",
    "origin",
    "dest",
    "route_key",
    "distance",
    "flight_date",

    "dep_hour_local",
    "dep_weekday_local",
    "dep_month_local",

    "dep_ts_sched_utc",
    "dep_ts_actual_utc",
    "arr_ts_sched_utc",
    "arr_ts_actual_utc",

    "dep_temp_c",
    "dep_wind_speed_m_s",
    "dep_wind_dir_deg",
    "dep_ceiling_height_m",
    "arr_temp_c",
    "arr_wind_speed_m_s",
    "arr_wind_dir_deg",
    "arr_ceiling_height_m",

    "prev_flight_id_same_tail",
    "next_flight_id_same_tail",
    "prev_origin",
    "prev_dest",
    "next_origin",
    "next_dest",
    "prev_arr_ts_actual_utc",
    "next_dep_ts_actual_utc",

    "dep_delay",
    "dep_del15",
    "arr_delay",
    "arr_del15",

    "prev_arr_delay",
    "prev_dep_delay",
    "next_arr_delay",
    "next_dep_delay",
    "prev_arr_del15",
    "prev_dep_del15",
    "next_dep_del15",
    "next_arr_del15",
    "prev_arr_late_15",
    "prev_dep_late_15",
    "next_arr_late_15",
    "next_dep_late_15",

    "turnaround_minutes",
    "next_turnaround_minutes",
    "rotation_continuity_flag",
    "next_rotation_continuity_flag",
    "aircraft_leg_number_day",
    "cum_dep_delay_aircraft_day",
    "cum_arr_delay_aircraft_day",

    "is_cancelled",
    "is_diverted",

    "crs_elapsed_time",
    "dep_time_blk",
    "arr_time_blk",
]

US_HOLIDAYS_2019 = [
    "2019-01-01",
    "2019-01-21",
    "2019-02-18",
    "2019-05-27",
    "2019-07-04",
    "2019-09-02",
    "2019-10-14",
    "2019-11-11",
    "2019-11-28",
    "2019-12-25",
]

ml_lf = (
    lf
    .select(RAW_FEATURES)
    .filter(
        (pl.col("is_cancelled") == 0) &
        (pl.col("is_diverted") == 0) &
        pl.col("arr_del15").is_not_null() &
        pl.col("tail_number").is_not_null() &
        pl.col("dep_ts_actual_utc").is_not_null() &
        pl.col("arr_ts_actual_utc").is_not_null()
    )
)

lf_features = (
    ml_lf
    .with_columns([
        pl.when(pl.col("dep_hour_local") < 6).then(1)
        .when(pl.col("dep_hour_local") < 11).then(2)
        .when(pl.col("dep_hour_local") < 14).then(3)
        .when(pl.col("dep_hour_local") < 18).then(4)
        .when(pl.col("dep_hour_local") < 21).then(5)
        .otherwise(6)
        .alias("dep_time_bucket"),

        pl.col("dep_weekday_local").is_in([6, 7]).cast(pl.Int8).alias("is_weekend"),

        pl.col("flight_date").cast(pl.Utf8).is_in(US_HOLIDAYS_2019).cast(pl.Int8).alias("is_holiday"),

        pl.min_horizontal([
            *[
                (pl.col("flight_date").cast(pl.Date) - pl.lit(h).str.strptime(pl.Date)).abs().dt.total_days()
                for h in US_HOLIDAYS_2019
            ]
        ]).alias("days_to_nearest_holiday"),

        pl.len().over("route_key").alias("route_frequency"),
        pl.len().over("origin").alias("origin_flight_volume"),
        pl.len().over("dest").alias("dest_flight_volume"),

        (pl.col("prev_arr_delay") > 15).cast(pl.Int8).alias("prev_arr_delayed_flag"),
        (pl.col("prev_arr_delay") + pl.col("prev_dep_delay")).alias("prev_total_delay"),

        (pl.col("turnaround_minutes") < 60).cast(pl.Int8).alias("tight_turnaround_flag"),

        (
            pl.col("aircraft_leg_number_day") /
            pl.max("aircraft_leg_number_day").over("tail_number")
        ).alias("relative_leg_position"),
    ])
)

usa_2hop_lf = (
    lf_features
    .sort(["tail_number", "dep_ts_actual_utc"])
    .with_columns([
        pl.col("flight_id").shift(1).over("tail_number").alias("prev1_flight_id"),
        pl.col("origin").shift(1).over("tail_number").alias("prev1_origin"),
        pl.col("dest").shift(1).over("tail_number").alias("prev1_dest"),
        pl.col("dep_ts_actual_utc").shift(1).over("tail_number").alias("prev1_dep_ts_utc"),
        pl.col("arr_ts_actual_utc").shift(1).over("tail_number").alias("prev1_arr_ts_utc"),
        pl.col("arr_delay").shift(1).over("tail_number").alias("prev1_arr_delay"),
        pl.col("dep_delay").shift(1).over("tail_number").alias("prev1_dep_delay"),
        pl.col("arr_del15").shift(1).over("tail_number").alias("prev1_arr_del15"),
        pl.col("dep_del15").shift(1).over("tail_number").alias("prev1_dep_del15"),

        pl.col("flight_id").shift(2).over("tail_number").alias("prev2_flight_id"),
        pl.col("origin").shift(2).over("tail_number").alias("prev2_origin"),
        pl.col("dest").shift(2).over("tail_number").alias("prev2_dest"),
        pl.col("dep_ts_actual_utc").shift(2).over("tail_number").alias("prev2_dep_ts_utc"),
        pl.col("arr_ts_actual_utc").shift(2).over("tail_number").alias("prev2_arr_ts_utc"),
        pl.col("arr_delay").shift(2).over("tail_number").alias("prev2_arr_delay"),
        pl.col("dep_delay").shift(2).over("tail_number").alias("prev2_dep_delay"),
        pl.col("arr_del15").shift(2).over("tail_number").alias("prev2_arr_del15"),
        pl.col("dep_del15").shift(2).over("tail_number").alias("prev2_dep_del15"),

        (
            pl.col("dep_ts_actual_utc") -
            pl.col("arr_ts_actual_utc").shift(1).over("tail_number")
        ).dt.total_minutes().alias("prev1_turnaround_minutes"),

        (
            pl.col("dep_ts_actual_utc") -
            pl.col("arr_ts_actual_utc").shift(2).over("tail_number")
        ).dt.total_minutes().alias("time_since_prev2_arrival_minutes"),
    ])
    .filter(
        pl.col("prev1_flight_id").is_not_null() &
        pl.col("prev2_flight_id").is_not_null() &
        pl.col("prev1_turnaround_minutes").is_not_null() &
        pl.col("time_since_prev2_arrival_minutes").is_not_null() &
        pl.col("prev1_turnaround_minutes").is_between(0, 12 * 60) &
        pl.col("time_since_prev2_arrival_minutes").is_between(0, 24 * 60)
    )
)

flights = usa_2hop_lf.collect()

flights = flights.with_columns(
    pl.col("dep_ts_actual_utc").cast(pl.Datetime).alias("dep_ts_actual_utc")
)

print("Rows in final modeling table:", flights.height)
timer_log("Feature engineering + collect", t0)


# ============================================================
# 3. FEATURE SETS
# ============================================================

xgb_full_features = [
    "distance",
    "dep_hour_local",
    "dep_weekday_local",
    "dep_month_local",
    "dep_time_bucket",
    "is_weekend",
    "is_holiday",
    "days_to_nearest_holiday",
    "crs_elapsed_time",

    "dep_temp_c",
    "dep_wind_speed_m_s",
    "dep_wind_dir_deg",
    "dep_ceiling_height_m",
    "arr_temp_c",
    "arr_wind_speed_m_s",
    "arr_wind_dir_deg",
    "arr_ceiling_height_m",
    "route_frequency",
    "origin_flight_volume",
    "dest_flight_volume",

    "prev_arr_delay",
    "prev_dep_delay",
    "prev_arr_del15",
    "prev_dep_del15",
    "prev_arr_delayed_flag",
    "prev_total_delay",
    "turnaround_minutes",
    "tight_turnaround_flag",
    "rotation_continuity_flag",
    "aircraft_leg_number_day",
    "relative_leg_position",
    "cum_dep_delay_aircraft_day",
    "cum_arr_delay_aircraft_day",
    "prev1_arr_delay",
    "prev1_dep_delay",
    "prev1_arr_del15",
    "prev1_dep_del15",
    "prev2_arr_delay",
    "prev2_dep_delay",
    "prev2_arr_del15",
    "prev2_dep_del15",
    "prev1_turnaround_minutes",
    "time_since_prev2_arrival_minutes",
]

lstm_context_current = [
    "distance",
    "dep_hour_local",
    "dep_weekday_local",
    "dep_month_local",
    "dep_time_bucket",
    "is_weekend",
    "is_holiday",
    "days_to_nearest_holiday",
    "crs_elapsed_time",

    "dep_temp_c",
    "dep_wind_speed_m_s",
    "dep_wind_dir_deg",
    "dep_ceiling_height_m",
    "arr_temp_c",
    "arr_wind_speed_m_s",
    "arr_wind_dir_deg",
    "arr_ceiling_height_m",
    "route_frequency",
    "origin_flight_volume",
    "dest_flight_volume",
    "tight_turnaround_flag",
    "relative_leg_position",
    "cum_dep_delay_aircraft_day",
    "cum_arr_delay_aircraft_day",
]

TARGET_CLASS = "arr_del15"
TARGET_REG = "arr_delay"


# ============================================================
# 4. FINAL TEST SPLIT
#    Keep Nov-Dec untouched
# ============================================================

t0 = time.perf_counter()

test_start_date = pd.Timestamp("2019-11-01")

train_tune_df = flights.filter(pl.col("dep_ts_actual_utc") < test_start_date)
test_df = flights.filter(pl.col("dep_ts_actual_utc") >= test_start_date)

print("Train+tune rows:", train_tune_df.height)
print("Final test rows:", test_df.height)

timer_log("Final train/test split", t0)


# ============================================================
# 5. HELPERS
# ============================================================

def safe_fill_and_to_pandas(df_pl: pl.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df_pl.select(cols).to_pandas()
    for c in out.columns:
        if out[c].dtype.kind in "biufc":
            med = out[c].median()
            if pd.isna(med):
                med = 0
            out[c] = out[c].fillna(med)
        else:
            out[c] = out[c].fillna("missing")
    return out

def classification_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "auc": roc_auc_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }

def regression_metrics(y_true, y_pred):
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
    }

def build_eval_table(results_rows):
    return pd.DataFrame(results_rows)

def summarize_cv_metrics(metric_list, prefix="cv"):
    keys = metric_list[0].keys()
    out = {}
    for k in keys:
        vals = np.array([m[k] for m in metric_list], dtype=float)
        out[f"{prefix}_{k}_mean"] = vals.mean()
        out[f"{prefix}_{k}_std"] = vals.std()
    return out

def rank_results(df):
    return (
        df.sort_values(
            by=["cv_auc_mean", "cv_f1_mean", "cv_mae_mean", "cv_rmse_mean"],
            ascending=[False, False, True, True]
        )
        .reset_index(drop=True)
    )


# ============================================================
# 6. TIME-AWARE CROSS-VALIDATION SPLITS
#    Expanding window CV inside Jan-Oct
# ============================================================

def make_time_cv_splits(df_pl: pl.DataFrame):
    """
    Expanding window folds inside Jan-Oct training period.

    Fold 1: train Jan-Jun, validate Jul
    Fold 2: train Jan-Jul, validate Aug
    Fold 3: train Jan-Aug, validate Sep
    Fold 4: train Jan-Sep, validate Oct
    """
    fold_specs = [
        ("2019-07-01", "2019-08-01"),
        ("2019-08-01", "2019-09-01"),
        ("2019-09-01", "2019-10-01"),
        ("2019-10-01", "2019-11-01"),
    ]

    splits = []
    for i, (val_start, val_end) in enumerate(fold_specs, start=1):
        val_start = pd.Timestamp(val_start)
        val_end = pd.Timestamp(val_end)

        train_fold = df_pl.filter(pl.col("dep_ts_actual_utc") < val_start)
        val_fold = df_pl.filter(
            (pl.col("dep_ts_actual_utc") >= val_start) &
            (pl.col("dep_ts_actual_utc") < val_end)
        )

        if train_fold.height == 0 or val_fold.height == 0:
            continue

        splits.append({
            "fold": i,
            "train_df": train_fold,
            "val_df": val_fold,
            "val_start": val_start,
            "val_end": val_end,
        })

    return splits

cv_splits = make_time_cv_splits(train_tune_df)
print(f"Built {len(cv_splits)} time-aware CV folds")
for s in cv_splits:
    print(
        f"Fold {s['fold']}: "
        f"train={s['train_df'].height:,} | "
        f"val={s['val_df'].height:,} | "
        f"val_window=[{s['val_start'].date()} -> {s['val_end'].date()})"
    )


# ============================================================
# 7. XGBOOST TUNING WITH TIME CV
# ============================================================

XGB_PARAM_GRID = [
    {
        "n_estimators": 250,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 1,
        "reg_lambda": 1.0,
    },
    {
        "n_estimators": 350,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 1.0,
        "min_child_weight": 1,
        "reg_lambda": 1.0,
    },
    {
        "n_estimators": 300,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 1,
        "reg_lambda": 1.0,
    },
    {
        "n_estimators": 400,
        "max_depth": 6,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 5,
        "reg_lambda": 5.0,
    },
    {
        "n_estimators": 300,
        "max_depth": 8,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 5,
        "reg_lambda": 5.0,
    },
]

def make_xgb_model(params, task="classification"):
    xgb_device = "cuda" if USE_GPU else "cpu"

    common = dict(
        random_state=42,
        n_jobs=-1,
        device=xgb_device,
        tree_method="hist",
        **params
    )

    if task == "classification":
        return XGBClassifier(
            eval_metric="logloss",
            **common
        )
    return XGBRegressor(**common)

def run_xgb_time_cv(cv_splits, feature_cols, param_grid):
    t_total = time.perf_counter()
    results_rows = []

    for config_id, params in enumerate(param_grid, start=1):
        t_cfg = time.perf_counter()

        cls_fold_metrics = []
        reg_fold_metrics = []

        for split in cv_splits:
            fold = split["fold"]
            train_fold = split["train_df"]
            val_fold = split["val_df"]

            X_train = safe_fill_and_to_pandas(train_fold, feature_cols)
            X_val = safe_fill_and_to_pandas(val_fold, feature_cols)

            y_train_cls = train_fold[TARGET_CLASS].to_pandas().astype(int)
            y_val_cls = val_fold[TARGET_CLASS].to_pandas().astype(int)

            y_train_reg = train_fold[TARGET_REG].to_pandas().astype(float)
            y_val_reg = val_fold[TARGET_REG].to_pandas().astype(float)

            clf = make_xgb_model(params, task="classification")
            clf.fit(X_train, y_train_cls)
            val_prob = clf.predict_proba(X_val)[:, 1]
            cls_metrics = classification_metrics(y_val_cls, val_prob)

            reg = make_xgb_model(params, task="regression")
            reg.fit(X_train, y_train_reg)
            val_pred_reg = reg.predict(X_val)
            reg_metrics = regression_metrics(y_val_reg, val_pred_reg)

            cls_fold_metrics.append(cls_metrics)
            reg_fold_metrics.append(reg_metrics)

            print(
                f"[XGB cfg {config_id:02d}] fold={fold} "
                f"AUC={cls_metrics['auc']:.4f} "
                f"F1={cls_metrics['f1']:.4f} "
                f"MAE={reg_metrics['mae']:.4f} "
                f"RMSE={reg_metrics['rmse']:.4f}"
            )

            del clf, reg, X_train, X_val
            gc.collect()

        row = {
            "model_family": "xgb_full_propagation",
            "config_id": f"xgb_{config_id:02d}",
            "params": deepcopy(params),
        }
        row.update(summarize_cv_metrics(cls_fold_metrics, prefix="cv"))
        row.update(summarize_cv_metrics(reg_fold_metrics, prefix="cv"))
        results_rows.append(row)

        timer_log(f"XGB config {config_id:02d}", t_cfg)

    results_df = rank_results(pd.DataFrame(results_rows))
    timer_log("Total XGB time-CV search", t_total)
    return results_df


# ============================================================
# 8. LSTM HELPERS
# ============================================================

STEP_FEATURES = [
    "arr_delay_prev",
    "dep_delay_prev",
    "arr_del15_prev",
    "dep_del15_prev",
    "gap_minutes",
    "distance",
    "dep_hour_local",
    "dep_weekday_local",
    "dep_month_local",
    "dep_time_bucket",
    "is_weekend",
    "is_holiday",
    "days_to_nearest_holiday",
    "crs_elapsed_time",
    "dep_temp_c",
    "dep_wind_speed_m_s",
    "dep_wind_dir_deg",
    "dep_ceiling_height_m",
    "arr_temp_c",
    "arr_wind_speed_m_s",
    "arr_wind_dir_deg",
    "arr_ceiling_height_m",
    "route_frequency",
    "origin_flight_volume",
    "dest_flight_volume",
    "tight_turnaround_flag",
    "relative_leg_position",
    "cum_dep_delay_aircraft_day",
    "cum_arr_delay_aircraft_day",
]

def build_lstm_step_matrix(df_pl: pl.DataFrame, current_variant="context_full"):
    """
    Example Matrix
    X = np.array([
        [
            [30, 25, 1, 1, 120],
            [45, 40, 1, 1,  75],
            [10,  5, 0, 0,   0]
        ],
        [
            [ 0,  0, 0, 0, 200],
            [ 5,  3, 0, 0, 150],
            [ 0,  0, 0, 0,   0]
        ],
        [
            [60, 50, 1, 1,  90],
            [70, 65, 1, 1,  60],
            [20, 15, 1, 1,   0]
        ]
        ], dtype=np.float32)

    Shape: 
    (3 samples, 3 time steps, 5 features)
    
    """
    pdf = df_pl.to_pandas().copy()

    numeric_fill_cols = [
        "prev2_arr_delay", "prev2_dep_delay", "prev2_arr_del15", "prev2_dep_del15",
        "prev1_arr_delay", "prev1_dep_delay", "prev1_arr_del15", "prev1_dep_del15",
        "prev1_turnaround_minutes", "time_since_prev2_arrival_minutes",
        "distance", "dep_hour_local", "dep_weekday_local", "dep_month_local",
        "dep_time_bucket", "is_weekend", "is_holiday", "days_to_nearest_holiday",
        "crs_elapsed_time", "dep_temp_c", "dep_wind_speed_m_s", "dep_wind_dir_deg",
        "dep_ceiling_height_m", "arr_temp_c", "arr_wind_speed_m_s", "arr_wind_dir_deg",
        "arr_ceiling_height_m", "route_frequency", "origin_flight_volume",
        "dest_flight_volume", "tight_turnaround_flag", "relative_leg_position",
        "cum_dep_delay_aircraft_day", "cum_arr_delay_aircraft_day"
    ]

    # This loop goes through selected numeric columns and fills any missing 
    # values with that column’s median (or 0 if the median itself is missing).
    for c in numeric_fill_cols:
        if c in pdf.columns:
            med = pdf[c].median()
            if pd.isna(med):
                med = 0
            pdf[c] = pdf[c].fillna(med) 

    current_context_cols = lstm_context_current

    # This creates a 3D NumPy array filled with zeros to hold LSTM input data, shaped as 
    # (number of samples, 3 time steps, number of features per step).
    X = np.zeros((len(pdf), 3, len(STEP_FEATURES)), dtype=np.float32)

    # This code is populating the first two time steps of the LSTM input tensor with features from previous flights.

    X[:, 0, STEP_FEATURES.index("arr_delay_prev")] = pdf["prev2_arr_delay"].values
    X[:, 0, STEP_FEATURES.index("dep_delay_prev")] = pdf["prev2_dep_delay"].values
    X[:, 0, STEP_FEATURES.index("arr_del15_prev")] = pdf["prev2_arr_del15"].values
    X[:, 0, STEP_FEATURES.index("dep_del15_prev")] = pdf["prev2_dep_del15"].values
    X[:, 0, STEP_FEATURES.index("gap_minutes")] = pdf["time_since_prev2_arrival_minutes"].values

    X[:, 1, STEP_FEATURES.index("arr_delay_prev")] = pdf["prev1_arr_delay"].values
    X[:, 1, STEP_FEATURES.index("dep_delay_prev")] = pdf["prev1_dep_delay"].values
    X[:, 1, STEP_FEATURES.index("arr_del15_prev")] = pdf["prev1_arr_del15"].values
    X[:, 1, STEP_FEATURES.index("dep_del15_prev")] = pdf["prev1_dep_del15"].values
    X[:, 1, STEP_FEATURES.index("gap_minutes")] = pdf["prev1_turnaround_minutes"].values

    for col in current_context_cols:
        if col in STEP_FEATURES:
            X[:, 2, STEP_FEATURES.index(col)] = pdf[col].values

    y_cls = pdf[TARGET_CLASS].astype(int).values
    y_reg = pdf[TARGET_REG].astype(float).values

    return X, y_cls, y_reg, pdf

def scale_lstm(X_train, X_val):
    """
    This function flattens the 3D LSTM input into 2D so a scaler can standardize each feature, then reshapes 
    it back to the original (samples, time steps, features) format after scaling. It fits the scaler only 
    the training data and applies the same transformation to validation data to avoid data leakage. 
    We need to do this because LSTMs (and neural networks in general) train more effectively when features 
    are on comparable scales, improving stability and convergence.
    """
    n_train, timesteps, n_features = X_train.shape
    n_val = X_val.shape[0]

    scaler = StandardScaler()
    X_train_2d = X_train.reshape(-1, n_features)
    X_val_2d = X_val.reshape(-1, n_features)

    X_train_scaled = scaler.fit_transform(X_train_2d).reshape(n_train, timesteps, n_features)
    X_val_scaled = scaler.transform(X_val_2d).reshape(n_val, timesteps, n_features)

    return X_train_scaled.astype(np.float32), X_val_scaled.astype(np.float32), scaler

def build_lstm_model(input_shape, units1=64, units2=32, dropout=0.2, learning_rate=1e-3):
    """
    
    """

    # This line defines the input layer of the model, telling Keras what shape each training 
    # example will have when it enters the network.
    # this creates a placeholder tensor where each sample is a sequence of feature 
    # vectors over time, which the LSTM layers will process next.
    inp = Input(shape=input_shape)

    # This applies the first LSTM layer to the input sequence, with units1 hidden units, meaning it learns a 
    # representation of the temporal patterns in the data.
    x = LSTM(units1, return_sequences=True)(inp)

    # This applies dropout regularization to the LSTM outputs, randomly turning off a fraction 
    # (dropout, e.g., 20%) of the neurons during training.
    x = Dropout(dropout)(x)

    # This applies a second LSTM layer that takes the full sequence from the first layer and compresses 
    # it into a single final representation of the entire sequence. In short, this layer turns the learned temporal 
    # patterns (delay propagation over time) into a compact feature vector the model can use for prediction.
    x = LSTM(units2)(x)
    x = Dropout(dropout)(x)

    ## Outputs

    # cls_out is a classification output that uses a sigmoid activation to predict a 
    # probability (e.g., likelihood of a 15+ minute delay).
    cls_out = Dense(1, activation="sigmoid", name="cls")(x)

    # reg_out is a regression output that uses a linear activation to predict a 
    # continuous value (e.g., actual delay minutes).
    reg_out = Dense(1, activation="linear", name="reg")(x)

    # This line assembles the full neural network by connecting the input layer (inp) 
    # to both output heads (cls_out and reg_out).
    model = Model(inputs=inp, outputs=[cls_out, reg_out])

    # This configures how the model will learn and be evaluated during training.
    # It uses the Adam optimizer to update weights efficiently during training.
        # Adam - (Adaptive Moment Estimation) it decides how big of a step to take when adjusting 
            # each parameter based on the gradients. Unlike basic gradient descent. This lets it adapt the 
            # learning rate for each parameter individually, making training faster and more stable.
    # It assigns different loss functions per output: binary cross-entropy for the classification head 
    # (cls) and mean squared error for the regression head (reg).
        # Binary cross-entropy is used for the classification output because it measures how well 
        # predicted probabilities match true 0/1 delay labels, while mean squared error is used for 
        # regression because it penalizes differences between predicted and actual delay minutes.
    # It tracks relevant metrics: AUC for classification performance and MAE (mean absolute error) for regression accuracy.
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss={"cls": "binary_crossentropy", "reg": "mse"},
        metrics={"cls": [tf.keras.metrics.AUC(name="auc")], "reg": ["mae"]},
    )
    return model


# ============================================================
# 9. LSTM TUNING WITH TIME CV
#    Keep this smaller because it is expensive
# ============================================================

LSTM_PARAM_GRID = [
    {
        "units1": 32,
        "units2": 16,
        "dropout": 0.2,
        "batch_size": 256,
        "epochs": 10,
        "learning_rate": 1e-3,
    },
    {
        "units1": 64,
        "units2": 32,
        "dropout": 0.2,
        "batch_size": 256,
        "epochs": 10,
        "learning_rate": 1e-3,
    },
    {
        "units1": 64,
        "units2": 32,
        "dropout": 0.3,
        "batch_size": 256,
        "epochs": 12,
        "learning_rate": 1e-3,
    },
    {
        "units1": 64,
        "units2": 32,
        "dropout": 0.2,
        "batch_size": 512,
        "epochs": 12,
        "learning_rate": 5e-4,
    },
]

def run_lstm_time_cv(cv_splits, param_grid):
    t_total = time.perf_counter()
    results_rows = []

    for config_id, params in enumerate(param_grid, start=1):
        t_cfg = time.perf_counter()

        cls_fold_metrics = []
        reg_fold_metrics = []

        for split in cv_splits:
            fold = split["fold"]
            train_fold = split["train_df"]
            val_fold = split["val_df"]

            X_train, y_train_cls, y_train_reg, _ = build_lstm_step_matrix(
                train_fold, current_variant="context_full"
            )
            X_val, y_val_cls, y_val_reg, _ = build_lstm_step_matrix(
                val_fold, current_variant="context_full"
            )

            X_train_scaled, X_val_scaled, scaler = scale_lstm(X_train, X_val)

            y_train_cls = np.asarray(y_train_cls, dtype=np.float32)
            y_train_reg = np.asarray(y_train_reg, dtype=np.float32)
            y_val_cls = np.asarray(y_val_cls, dtype=np.float32)
            y_val_reg = np.asarray(y_val_reg, dtype=np.float32)

            tf.keras.backend.clear_session()

            model = build_lstm_model(
                input_shape=(X_train_scaled.shape[1], X_train_scaled.shape[2]),
                units1=params["units1"],
                units2=params["units2"],
                dropout=params["dropout"],
                learning_rate=params["learning_rate"],
            )

            early_stop = EarlyStopping(
                monitor="val_loss",
                patience=3,
                restore_best_weights=True
            )

            model.fit(
                X_train_scaled,
                {"cls": y_train_cls, "reg": y_train_reg},
                validation_data=(X_val_scaled, {"cls": y_val_cls, "reg": y_val_reg}),
                epochs=params["epochs"],
                batch_size=params["batch_size"],
                callbacks=[early_stop],
                verbose=0,
            )

            pred_cls, pred_reg = model.predict(X_val_scaled, verbose=0)
            pred_cls = pred_cls.ravel()
            pred_reg = pred_reg.ravel()

            cls_metrics = classification_metrics(y_val_cls.astype(int), pred_cls)
            reg_metrics = regression_metrics(y_val_reg, pred_reg)

            cls_fold_metrics.append(cls_metrics)
            reg_fold_metrics.append(reg_metrics)

            print(
                f"[LSTM cfg {config_id:02d}] fold={fold} "
                f"AUC={cls_metrics['auc']:.4f} "
                f"F1={cls_metrics['f1']:.4f} "
                f"MAE={reg_metrics['mae']:.4f} "
                f"RMSE={reg_metrics['rmse']:.4f}"
            )

            del model, X_train, X_val, X_train_scaled, X_val_scaled, scaler
            gc.collect()

        row = {
            "model_family": "lstm_context_full",
            "config_id": f"lstm_{config_id:02d}",
            "params": deepcopy(params),
        }
        row.update(summarize_cv_metrics(cls_fold_metrics, prefix="cv"))
        row.update(summarize_cv_metrics(reg_fold_metrics, prefix="cv"))
        results_rows.append(row)

        timer_log(f"LSTM config {config_id:02d}", t_cfg)

    results_df = rank_results(pd.DataFrame(results_rows))
    timer_log("Total LSTM time-CV search", t_total)
    return results_df


# ============================================================
# 10. RUN TUNING
# ============================================================

t0 = time.perf_counter()
xgb_cv_results = run_xgb_time_cv(
    cv_splits=cv_splits,
    feature_cols=xgb_full_features,
    param_grid=XGB_PARAM_GRID,
)
timer_log("XGB CV search block", t0)

t0 = time.perf_counter()
lstm_cv_results = run_lstm_time_cv(
    cv_splits=cv_splits,
    param_grid=LSTM_PARAM_GRID,
)
timer_log("LSTM CV search block", t0)

print("\nTop XGB configs")
print(xgb_cv_results.head(10))

print("\nTop LSTM configs")
print(lstm_cv_results.head(10))

best_xgb_row = xgb_cv_results.iloc[0].to_dict()
best_lstm_row = lstm_cv_results.iloc[0].to_dict()

best_family_df = pd.DataFrame([best_xgb_row, best_lstm_row]).sort_values(
    by=["cv_auc_mean", "cv_f1_mean", "cv_mae_mean", "cv_rmse_mean"],
    ascending=[False, False, True, True]
).reset_index(drop=True)

print("\nBest config per family")
print(best_family_df[[
    "model_family", "config_id",
    "cv_auc_mean", "cv_f1_mean", "cv_precision_mean", "cv_recall_mean", "cv_accuracy_mean",
    "cv_mae_mean", "cv_rmse_mean", "params"
]])


# ============================================================
# 11. FINAL REFIT ON JAN-OCT, EVALUATE ON NOV-DEC
# ============================================================

def refit_best_xgb(train_df, test_df, feature_cols, best_params):
    X_train = safe_fill_and_to_pandas(train_df, feature_cols)
    X_test = safe_fill_and_to_pandas(test_df, feature_cols)

    y_train_cls = train_df[TARGET_CLASS].to_pandas().astype(int)
    y_test_cls = test_df[TARGET_CLASS].to_pandas().astype(int)

    y_train_reg = train_df[TARGET_REG].to_pandas().astype(float)
    y_test_reg = test_df[TARGET_REG].to_pandas().astype(float)

    clf = make_xgb_model(best_params, task="classification")
    clf.fit(X_train, y_train_cls)
    test_prob = clf.predict_proba(X_test)[:, 1]

    reg = make_xgb_model(best_params, task="regression")
    reg.fit(X_train, y_train_reg)
    test_pred_reg = reg.predict(X_test)

    return {
        "model_family": "xgb_full_propagation",
        "params": deepcopy(best_params),
        "classifier": clf,
        "regressor": reg,
        "features": feature_cols,
        "test_pred_cls": test_prob,
        "test_pred_reg": test_pred_reg,
        "test_cls_metrics": classification_metrics(y_test_cls, test_prob),
        "test_reg_metrics": regression_metrics(y_test_reg, test_pred_reg),
        "y_test_cls": y_test_cls,
        "y_test_reg": y_test_reg,
    }

def refit_best_lstm(train_df, test_df, best_params):
    X_train, y_train_cls, y_train_reg, train_pdf = build_lstm_step_matrix(
        train_df, current_variant="context_full"
    )
    X_test, y_test_cls, y_test_reg, test_pdf = build_lstm_step_matrix(
        test_df, current_variant="context_full"
    )

    X_train_scaled, X_test_scaled, scaler = scale_lstm(X_train, X_test)

    y_train_cls = np.asarray(y_train_cls, dtype=np.float32)
    y_train_reg = np.asarray(y_train_reg, dtype=np.float32)
    y_test_cls = np.asarray(y_test_cls, dtype=np.float32)
    y_test_reg = np.asarray(y_test_reg, dtype=np.float32)

    tf.keras.backend.clear_session()

    model = build_lstm_model(
        input_shape=(X_train_scaled.shape[1], X_train_scaled.shape[2]),
        units1=best_params["units1"],
        units2=best_params["units2"],
        dropout=best_params["dropout"],
        learning_rate=best_params["learning_rate"],
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    model.fit(
        X_train_scaled,
        {"cls": y_train_cls, "reg": y_train_reg},
        validation_split=0.10,
        epochs=best_params["epochs"],
        batch_size=best_params["batch_size"],
        callbacks=[early_stop],
        verbose=1,
    )

    pred_cls, pred_reg = model.predict(X_test_scaled, verbose=0)
    pred_cls = pred_cls.ravel()
    pred_reg = pred_reg.ravel()

    return {
        "model_family": "lstm_context_full",
        "params": deepcopy(best_params),
        "model": model,
        "scaler": scaler,
        "test_pdf": test_pdf,
        "X_test": X_test_scaled,
        "test_pred_cls": pred_cls,
        "test_pred_reg": pred_reg,
        "test_cls_metrics": classification_metrics(y_test_cls.astype(int), pred_cls),
        "test_reg_metrics": regression_metrics(y_test_reg, pred_reg),
        "y_test_cls": y_test_cls,
        "y_test_reg": y_test_reg,
    }

t0 = time.perf_counter()

best_xgb_final = refit_best_xgb(
    train_df=train_tune_df,
    test_df=test_df,
    feature_cols=xgb_full_features,
    best_params=best_xgb_row["params"],
)

best_lstm_final = refit_best_lstm(
    train_df=train_tune_df,
    test_df=test_df,
    best_params=best_lstm_row["params"],
)

timer_log("Final refit on Jan-Oct + test on Nov-Dec", t0)


# ============================================================
# 12. FINAL TEST RESULTS
# ============================================================

final_test_results = pd.DataFrame([
    {
        "model_family": best_xgb_final["model_family"],
        "params": best_xgb_final["params"],
        "test_auc": best_xgb_final["test_cls_metrics"]["auc"],
        "test_f1": best_xgb_final["test_cls_metrics"]["f1"],
        "test_precision": best_xgb_final["test_cls_metrics"]["precision"],
        "test_recall": best_xgb_final["test_cls_metrics"]["recall"],
        "test_accuracy": best_xgb_final["test_cls_metrics"]["accuracy"],
        "test_mae": best_xgb_final["test_reg_metrics"]["mae"],
        "test_rmse": best_xgb_final["test_reg_metrics"]["rmse"],
    },
    {
        "model_family": best_lstm_final["model_family"],
        "params": best_lstm_final["params"],
        "test_auc": best_lstm_final["test_cls_metrics"]["auc"],
        "test_f1": best_lstm_final["test_cls_metrics"]["f1"],
        "test_precision": best_lstm_final["test_cls_metrics"]["precision"],
        "test_recall": best_lstm_final["test_cls_metrics"]["recall"],
        "test_accuracy": best_lstm_final["test_cls_metrics"]["accuracy"],
        "test_mae": best_lstm_final["test_reg_metrics"]["mae"],
        "test_rmse": best_lstm_final["test_reg_metrics"]["rmse"],
    },
]).sort_values(
    by=["test_auc", "test_f1", "test_mae", "test_rmse"],
    ascending=[False, False, True, True]
).reset_index(drop=True)

print("\n================================================")
print("FINAL TEST RESULTS")
print("================================================")
print(final_test_results)

best_overall_model = final_test_results.iloc[0].to_dict()

print("\n================================================")
print("BEST OVERALL MODEL")
print("================================================")
print(best_overall_model)


# ============================================================
# 13. ROC CURVES
# ============================================================

plt.figure(figsize=(8, 6))

xgb_prob = best_xgb_final["test_pred_cls"]
lstm_prob = best_lstm_final["test_pred_cls"]
y_test_cls = np.asarray(best_xgb_final["y_test_cls"]).astype(int)

for name, probs in [
    ("xgb_full_propagation", xgb_prob),
    ("lstm_context_full", lstm_prob),
]:
    fpr, tpr, _ = roc_curve(y_test_cls, probs)
    auc_val = roc_auc_score(y_test_cls, probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves: Final Tuned Models on Nov-Dec Test")
plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# 14. ACTUAL VS PREDICTED DELAY
# ============================================================

best_name = final_test_results.iloc[0]["model_family"]

if best_name == "xgb_full_propagation":
    y_true_reg = np.asarray(best_xgb_final["y_test_reg"])
    y_pred_reg = np.asarray(best_xgb_final["test_pred_reg"])
else:
    y_true_reg = np.asarray(best_lstm_final["y_test_reg"])
    y_pred_reg = np.asarray(best_lstm_final["test_pred_reg"])

sample_n = min(5000, len(y_true_reg))
idx = np.random.RandomState(42).choice(len(y_true_reg), size=sample_n, replace=False)

plt.figure(figsize=(8, 6))
plt.scatter(y_true_reg[idx], y_pred_reg[idx], alpha=0.3)
plt.xlabel("Actual Arrival Delay (minutes)")
plt.ylabel("Predicted Arrival Delay (minutes)")
plt.title(f"Actual vs Predicted Delay: {best_name}")
plt.tight_layout()
plt.show()


# ============================================================
# 15. FEATURE IMPORTANCE FOR BEST XGB
# ============================================================

importance_df = pd.DataFrame({
    "feature": best_xgb_final["features"],
    "importance": best_xgb_final["classifier"].feature_importances_
}).sort_values("importance", ascending=False).head(20)

plt.figure(figsize=(10, 7))
plt.barh(importance_df["feature"][::-1], importance_df["importance"][::-1])
plt.title("Top Feature Importances: Tuned xgb_full_propagation")
plt.tight_layout()
plt.show()


# ============================================================
# 16. SAVE RESULTS OBJECT
# ============================================================

tuned_model_results = {
    "cv_splits": [
        {
            "fold": s["fold"],
            "val_start": str(s["val_start"].date()),
            "val_end": str(s["val_end"].date()),
            "train_rows": s["train_df"].height,
            "val_rows": s["val_df"].height,
        }
        for s in cv_splits
    ],
    "xgb_cv_results": xgb_cv_results,
    "lstm_cv_results": lstm_cv_results,
    "best_xgb_row": best_xgb_row,
    "best_lstm_row": best_lstm_row,
    "best_xgb_final": best_xgb_final,
    "best_lstm_final": best_lstm_final,
    "final_test_results": final_test_results,
    "best_overall_model": best_overall_model,
}

print("\nSaved results to `tuned_model_results`")
timer_log("TOTAL PIPELINE TIME", GLOBAL_START)

[SYSTEM] Detected 24 CPU cores


2026-04-08 08:29:39.148925: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-08 08:29:39.154923: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775651379.162142  138341 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775651379.164629  138341 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775651379.170684  138341 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

[TF] inter_op threads: 0
[TF] intra_op threads: 0
[TF] CPU-only mode enabled.
[TIMER] Load data: 0.02s
Rows in final modeling table: 6594553
[TIMER] Feature engineering + collect: 2.68s
Train+tune rows: 5490189
Final test rows: 1104364
[TIMER] Final train/test split: 0.40s
Built 4 time-aware CV folds
Fold 1: train=3,191,329 | val=588,248 | val_window=[2019-07-01 -> 2019-08-01)
Fold 2: train=3,779,577 | val=594,508 | val_window=[2019-08-01 -> 2019-09-01)
Fold 3: train=4,374,085 | val=537,191 | val_window=[2019-09-01 -> 2019-10-01)
Fold 4: train=4,911,276 | val=578,913 | val_window=[2019-10-01 -> 2019-11-01)
[XGB cfg 01] fold=1 AUC=0.9798 F1=0.8439 MAE=8.8601 RMSE=20.0685
[XGB cfg 01] fold=2 AUC=0.9803 F1=0.8466 MAE=8.5582 RMSE=19.4063
[XGB cfg 01] fold=3 AUC=0.9828 F1=0.8232 MAE=7.1096 RMSE=16.3279
[XGB cfg 01] fold=4 AUC=0.9802 F1=0.8266 MAE=7.4529 RMSE=17.0205
[TIMER] XGB config 01: 109.12s
[XGB cfg 02] fold=1 AUC=0.9827 F1=0.8570 MAE=8.3418 RMSE=19.2936
[XGB cfg 02] fold=2 AUC=0.9825

I0000 00:00:1775652291.743746  150240 service.cc:152] XLA service 0x706b7000f8b0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775652291.743769  150240 service.cc:160]   StreamExecutor device (0): Host, Default Version
2026-04-08 08:44:51.778035: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1775652292.260702  150240 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


[LSTM cfg 01] fold=1 AUC=0.9760 F1=0.8360 MAE=7.1912 RMSE=18.4797
[LSTM cfg 01] fold=2 AUC=0.9747 F1=0.8415 MAE=7.5161 RMSE=17.7543
[LSTM cfg 01] fold=3 AUC=0.9807 F1=0.8204 MAE=5.7616 RMSE=14.2313
[LSTM cfg 01] fold=4 AUC=0.9758 F1=0.8264 MAE=6.1722 RMSE=14.6968
[TIMER] LSTM config 01: 1400.59s
[LSTM cfg 02] fold=1 AUC=0.9799 F1=0.8735 MAE=6.7946 RMSE=16.2032
[LSTM cfg 02] fold=2 AUC=0.9789 F1=0.8756 MAE=7.2892 RMSE=16.3747
[LSTM cfg 02] fold=3 AUC=0.9818 F1=0.8439 MAE=5.4865 RMSE=13.4848
[LSTM cfg 02] fold=4 AUC=0.9774 F1=0.8180 MAE=5.6279 RMSE=13.8305
[TIMER] LSTM config 02: 2293.40s
[LSTM cfg 03] fold=1 AUC=0.9772 F1=0.8640 MAE=7.1113 RMSE=16.1763
[LSTM cfg 03] fold=2 AUC=0.9788 F1=0.8387 MAE=6.5939 RMSE=16.2910
[LSTM cfg 03] fold=3 AUC=0.9802 F1=0.8297 MAE=5.7421 RMSE=13.5845


# Claude 

In [ ]:
# ============================================================
# Flight Delay — XGBoost vs LSTM
# Grid Search Hyperparameter Tuning + Walk-Forward CV
#
# Strategy overview
# -----------------
# This data has strong temporal autocorrelation (delay propagation
# along tail-number chains) so standard k-fold CV would leak future
# information into training.  We use *expanding-window walk-forward
# CV* instead:
#
#   Fold 1  train Jan–Jun   │ validate Jul–Aug   (grid-search fold)
#   Fold 2  train Jan–Aug   │ validate Sep–Oct
#   Fold 3  train Jan–Oct   │ validate Nov–Dec   (matches original split)
#
# Grid search runs on Fold 1 only (cheapest).  The winning params are
# then re-evaluated on all 3 folds to measure seasonal stability.
# A final model is retrained on Jan–Oct and evaluated on Nov–Dec.
# ============================================================

from pathlib import Path
import sys
import importlib
import warnings
import itertools
import json
warnings.filterwarnings("ignore")

import os
import time

# ============================================================
# MULTI-CORE SETUP
# ============================================================
import multiprocessing
N_CORES = multiprocessing.cpu_count()
print(f"[SYSTEM] Detected {N_CORES} CPU cores")

os.environ["OMP_NUM_THREADS"]        = str(N_CORES)
os.environ["MKL_NUM_THREADS"]        = str(N_CORES)
os.environ["OPENBLAS_NUM_THREADS"]   = str(N_CORES)
os.environ["NUMEXPR_NUM_THREADS"]    = str(N_CORES)
os.environ["TF_NUM_INTEROP_THREADS"] = str(N_CORES)
os.environ["TF_NUM_INTRAOP_THREADS"] = str(N_CORES)

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    accuracy_score, mean_absolute_error, mean_squared_error, roc_curve,
)
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier, XGBRegressor

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print(f"[TF] inter_op  threads: {tf.config.threading.get_inter_op_parallelism_threads()}")
print(f"[TF] intra_op  threads: {tf.config.threading.get_intra_op_parallelism_threads()}")

# ============================================================
# GLOBAL CONFIG  ← edit here to trade speed for coverage
# ============================================================

USE_GPU   = True          # set False to force CPU
FAST_MODE = False         # True → tiny grids for smoke-testing

# Walk-forward folds.  Each entry: (train_end_excl, val_end_excl)
# Month boundaries are [inclusive start = Jan 1, exclusive end).
# Fold 1 is also the grid-search fold.
CV_FOLDS = [
    # label           train_end (excl)     val_end (excl)
    ("Fold1 val Jul–Aug", "2019-07-01",    "2019-09-01"),
    ("Fold2 val Sep–Oct", "2019-09-01",    "2019-11-01"),
    ("Fold3 val Nov–Dec", "2019-11-01",    "2020-01-01"),   # = original holdout
]
# All folds share the same training start
TRAIN_START = "2019-01-01"

LSTM_EPOCHS    = 15    # hard cap; EarlyStopping (patience=3) will cut short
LSTM_VAL_SPLIT = 0.15  # fraction of training data used as Keras val set

# ============================================================
# TIMER HELPERS
# ============================================================

GLOBAL_START = time.perf_counter()

def timer_log(label: str, t0: float) -> None:
    print(f"[TIMER] {label}: {time.perf_counter() - t0:.1f}s")

def section(title: str) -> None:
    print(f"\n{'='*64}")
    print(f"  {title}")
    print(f"{'='*64}")


# ============================================================
# TENSORFLOW DEVICE SETUP
# ============================================================

try:
    if USE_GPU:
        gpus = tf.config.list_physical_devices("GPU")
        if gpus:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"[TF] GPU mode — {len(gpus)} device(s): {[g.name for g in gpus]}")
        else:
            print("[TF] WARNING: USE_GPU=True but no GPU detected — using CPU.")
    else:
        tf.config.set_visible_devices([], "GPU")
        print("[TF] CPU-only mode.")
except RuntimeError as e:
    print(f"[TF WARNING] {e}")


# ============================================================
# 1. LOAD DATA
# ============================================================

section("1. LOAD DATA")
t0 = time.perf_counter()

pl.Config.set_tbl_rows(1000)
pl.Config.set_tbl_cols(100)
pl.Config.set_tbl_width_chars(200)

current = Path.cwd()
while current.name != "shared-notebooks":
    if current.parent == current:
        raise RuntimeError("Could not locate shared-notebooks directory")
    current = current.parent

utils_path = current / "common_utils" / "python"
if str(utils_path) not in sys.path:
    sys.path.append(str(utils_path))

import load_flight_data
importlib.reload(load_flight_data)
lf = load_flight_data.load_flight_data(file_name="flights_canonical_2019.parquet")

timer_log("Load data", t0)


# ============================================================
# 2. FEATURE ENGINEERING  (unchanged from baseline)
# ============================================================

section("2. FEATURE ENGINEERING")
t0 = time.perf_counter()

RAW_FEATURES = [
    "flight_id", "tail_number", "reporting_airline", "origin", "dest",
    "route_key", "distance", "flight_date",
    "dep_hour_local", "dep_weekday_local", "dep_month_local",
    "dep_ts_sched_utc", "dep_ts_actual_utc", "arr_ts_sched_utc", "arr_ts_actual_utc",
    "dep_temp_c", "dep_wind_speed_m_s", "dep_wind_dir_deg", "dep_ceiling_height_m",
    "arr_temp_c", "arr_wind_speed_m_s", "arr_wind_dir_deg", "arr_ceiling_height_m",
    "prev_flight_id_same_tail", "next_flight_id_same_tail",
    "prev_origin", "prev_dest", "next_origin", "next_dest",
    "prev_arr_ts_actual_utc", "next_dep_ts_actual_utc",
    "dep_delay", "dep_del15", "arr_delay", "arr_del15",
    "prev_arr_delay", "prev_dep_delay", "next_arr_delay", "next_dep_delay",
    "prev_arr_del15", "prev_dep_del15", "next_dep_del15", "next_arr_del15",
    "prev_arr_late_15", "prev_dep_late_15", "next_arr_late_15", "next_dep_late_15",
    "turnaround_minutes", "next_turnaround_minutes",
    "rotation_continuity_flag", "next_rotation_continuity_flag",
    "aircraft_leg_number_day", "cum_dep_delay_aircraft_day", "cum_arr_delay_aircraft_day",
    "is_cancelled", "is_diverted",
    "crs_elapsed_time", "dep_time_blk", "arr_time_blk",
]

US_HOLIDAYS_2019 = [
    "2019-01-01", "2019-01-21", "2019-02-18", "2019-05-27",
    "2019-07-04", "2019-09-02", "2019-10-14", "2019-11-11",
    "2019-11-28", "2019-12-25",
]

ml_lf = (
    lf.select(RAW_FEATURES)
    .filter(
        (pl.col("is_cancelled") == 0) &
        (pl.col("is_diverted") == 0) &
        pl.col("arr_del15").is_not_null() &
        pl.col("tail_number").is_not_null() &
        pl.col("dep_ts_actual_utc").is_not_null() &
        pl.col("arr_ts_actual_utc").is_not_null()
    )
)

lf_features = ml_lf.with_columns([
    pl.when(pl.col("dep_hour_local") < 6).then(1)
      .when(pl.col("dep_hour_local") < 11).then(2)
      .when(pl.col("dep_hour_local") < 14).then(3)
      .when(pl.col("dep_hour_local") < 18).then(4)
      .when(pl.col("dep_hour_local") < 21).then(5)
      .otherwise(6).alias("dep_time_bucket"),
    pl.col("dep_weekday_local").is_in([6, 7]).cast(pl.Int8).alias("is_weekend"),
    pl.col("flight_date").cast(pl.Utf8).is_in(US_HOLIDAYS_2019).cast(pl.Int8).alias("is_holiday"),
    pl.min_horizontal([
        (pl.col("flight_date").cast(pl.Date) - pl.lit(h).str.strptime(pl.Date)).abs().dt.total_days()
        for h in US_HOLIDAYS_2019
    ]).alias("days_to_nearest_holiday"),
    pl.len().over("route_key").alias("route_frequency"),
    pl.len().over("origin").alias("origin_flight_volume"),
    pl.len().over("dest").alias("dest_flight_volume"),
    (pl.col("prev_arr_delay") > 15).cast(pl.Int8).alias("prev_arr_delayed_flag"),
    (pl.col("prev_arr_delay") + pl.col("prev_dep_delay")).alias("prev_total_delay"),
    (pl.col("turnaround_minutes") < 60).cast(pl.Int8).alias("tight_turnaround_flag"),
    (
        pl.col("aircraft_leg_number_day") /
        pl.max("aircraft_leg_number_day").over("tail_number")
    ).alias("relative_leg_position"),
])

flights = (
    lf_features
    .sort(["tail_number", "dep_ts_actual_utc"])
    .with_columns([
        # 1-hop back
        pl.col("flight_id").shift(1).over("tail_number").alias("prev1_flight_id"),
        pl.col("origin").shift(1).over("tail_number").alias("prev1_origin"),
        pl.col("dest").shift(1).over("tail_number").alias("prev1_dest"),
        pl.col("arr_delay").shift(1).over("tail_number").alias("prev1_arr_delay"),
        pl.col("dep_delay").shift(1).over("tail_number").alias("prev1_dep_delay"),
        pl.col("arr_del15").shift(1).over("tail_number").alias("prev1_arr_del15"),
        pl.col("dep_del15").shift(1).over("tail_number").alias("prev1_dep_del15"),
        pl.col("arr_ts_actual_utc").shift(1).over("tail_number").alias("prev1_arr_ts_utc"),
        # 2-hops back
        pl.col("flight_id").shift(2).over("tail_number").alias("prev2_flight_id"),
        pl.col("origin").shift(2).over("tail_number").alias("prev2_origin"),
        pl.col("dest").shift(2).over("tail_number").alias("prev2_dest"),
        pl.col("arr_delay").shift(2).over("tail_number").alias("prev2_arr_delay"),
        pl.col("dep_delay").shift(2).over("tail_number").alias("prev2_dep_delay"),
        pl.col("arr_del15").shift(2).over("tail_number").alias("prev2_arr_del15"),
        pl.col("dep_del15").shift(2).over("tail_number").alias("prev2_dep_del15"),
        pl.col("arr_ts_actual_utc").shift(2).over("tail_number").alias("prev2_arr_ts_utc"),
        # timing gaps
        (
            pl.col("dep_ts_actual_utc") -
            pl.col("arr_ts_actual_utc").shift(1).over("tail_number")
        ).dt.total_minutes().alias("prev1_turnaround_minutes"),
        (
            pl.col("dep_ts_actual_utc") -
            pl.col("arr_ts_actual_utc").shift(2).over("tail_number")
        ).dt.total_minutes().alias("time_since_prev2_arrival_minutes"),
    ])
    .filter(
        pl.col("prev1_flight_id").is_not_null() &
        pl.col("prev2_flight_id").is_not_null() &
        pl.col("prev1_turnaround_minutes").is_not_null() &
        pl.col("time_since_prev2_arrival_minutes").is_not_null() &
        pl.col("prev1_turnaround_minutes").is_between(0, 12 * 60) &
        pl.col("time_since_prev2_arrival_minutes").is_between(0, 24 * 60)
    )
    .with_columns(
        pl.col("dep_ts_actual_utc").cast(pl.Datetime)
    )
    .collect()
)

print(f"Modeling table rows: {flights.height:,}")
timer_log("Feature engineering + collect", t0)


# ============================================================
# 3. FEATURE DEFINITIONS
# ============================================================

XGB_FEATURES = [
    "distance", "dep_hour_local", "dep_weekday_local", "dep_month_local",
    "dep_time_bucket", "is_weekend", "is_holiday", "days_to_nearest_holiday",
    "crs_elapsed_time",
    "dep_temp_c", "dep_wind_speed_m_s", "dep_wind_dir_deg", "dep_ceiling_height_m",
    "arr_temp_c", "arr_wind_speed_m_s", "arr_wind_dir_deg", "arr_ceiling_height_m",
    "route_frequency", "origin_flight_volume", "dest_flight_volume",
    "prev_arr_delay", "prev_dep_delay", "prev_arr_del15", "prev_dep_del15",
    "prev_arr_delayed_flag", "prev_total_delay",
    "turnaround_minutes", "tight_turnaround_flag", "rotation_continuity_flag",
    "aircraft_leg_number_day", "relative_leg_position",
    "cum_dep_delay_aircraft_day", "cum_arr_delay_aircraft_day",
    "prev1_arr_delay", "prev1_dep_delay", "prev1_arr_del15", "prev1_dep_del15",
    "prev2_arr_delay", "prev2_dep_delay", "prev2_arr_del15", "prev2_dep_del15",
    "prev1_turnaround_minutes", "time_since_prev2_arrival_minutes",
]

# Features placed at the *current* step slot in the LSTM sequence
LSTM_CURRENT_FEATURES = [
    "distance", "dep_hour_local", "dep_weekday_local", "dep_month_local",
    "dep_time_bucket", "is_weekend", "is_holiday", "days_to_nearest_holiday",
    "crs_elapsed_time",
    "dep_temp_c", "dep_wind_speed_m_s", "dep_wind_dir_deg", "dep_ceiling_height_m",
    "arr_temp_c", "arr_wind_speed_m_s", "arr_wind_dir_deg", "arr_ceiling_height_m",
    "route_frequency", "origin_flight_volume", "dest_flight_volume",
    "tight_turnaround_flag", "relative_leg_position",
    "cum_dep_delay_aircraft_day", "cum_arr_delay_aircraft_day",
]

# Full ordered feature list for LSTM step axis
STEP_FEATURES = [
    "arr_delay_prev", "dep_delay_prev", "arr_del15_prev", "dep_del15_prev",
    "gap_minutes",
    "distance", "dep_hour_local", "dep_weekday_local", "dep_month_local",
    "dep_time_bucket", "is_weekend", "is_holiday", "days_to_nearest_holiday",
    "crs_elapsed_time",
    "dep_temp_c", "dep_wind_speed_m_s", "dep_wind_dir_deg", "dep_ceiling_height_m",
    "arr_temp_c", "arr_wind_speed_m_s", "arr_wind_dir_deg", "arr_ceiling_height_m",
    "route_frequency", "origin_flight_volume", "dest_flight_volume",
    "tight_turnaround_flag", "relative_leg_position",
    "cum_dep_delay_aircraft_day", "cum_arr_delay_aircraft_day",
]

TARGET_CLASS = "arr_del15"
TARGET_REG   = "arr_delay"

# Numeric cols to median-fill in LSTM matrix builder
LSTM_NUMERIC_FILL = [
    "prev2_arr_delay", "prev2_dep_delay", "prev2_arr_del15", "prev2_dep_del15",
    "prev1_arr_delay", "prev1_dep_delay", "prev1_arr_del15", "prev1_dep_del15",
    "prev1_turnaround_minutes", "time_since_prev2_arrival_minutes",
] + LSTM_CURRENT_FEATURES


# ============================================================
# 4. HYPERPARAMETER GRIDS
# ============================================================

# XGBoost — 24 combos.  Covers depth, shrinkage, and tree diversity.
# Deliberately avoids n_estimators sweep (controlled with early_stopping_rounds
# inside the fit call instead — cheaper and more principled).
XGB_PARAM_GRID = {
    "max_depth":        [4, 6, 8],
    "learning_rate":    [0.03, 0.07],
    "subsample":        [0.75, 0.90],
    "colsample_bytree": [0.75, 0.90],
}

# LSTM — 12 combos.  Width and regularisation are most impactful;
# batch size and lr are held to sensible defaults to keep cost manageable.
LSTM_PARAM_GRID = {
    "units1":        [64, 128],
    "units2":        [32, 64],
    "dropout":       [0.1, 0.2, 0.3],
}
LSTM_LR         = 1e-3   # fixed Adam lr
LSTM_BATCH_SIZE = 256    # fixed batch size

if FAST_MODE:
    XGB_PARAM_GRID = {
        "max_depth":        [4, 6],
        "learning_rate":    [0.07],
        "subsample":        [0.80],
        "colsample_bytree": [0.80],
    }
    LSTM_PARAM_GRID = {
        "units1":   [64],
        "units2":   [32],
        "dropout":  [0.2],
    }


# ============================================================
# 5. SHARED UTILITY FUNCTIONS
# ============================================================

def fold_split(
    df: pl.DataFrame,
    train_end: str,
    val_end: str,
    train_start: str = TRAIN_START,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """
    Return (train_df, val_df) for one walk-forward fold.
    Training always starts from TRAIN_START so the window expands.
    """
    t_start = pd.Timestamp(train_start)
    t_end   = pd.Timestamp(train_end)
    v_end   = pd.Timestamp(val_end)
    train = df.filter(
        (pl.col("dep_ts_actual_utc") >= t_start) &
        (pl.col("dep_ts_actual_utc") <  t_end)
    )
    val = df.filter(
        (pl.col("dep_ts_actual_utc") >= t_end) &
        (pl.col("dep_ts_actual_utc") <  v_end)
    )
    return train, val


def safe_fill(df_pl: pl.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df_pl.select(cols).to_pandas()
    for c in out.columns:
        if out[c].dtype.kind in "biufc":
            out[c] = out[c].fillna(out[c].median())
        else:
            out[c] = out[c].fillna("missing")
    return out


def cls_metrics(y_true, y_prob, thr: float = 0.5) -> dict:
    yp = (y_prob >= thr).astype(int)
    return {
        "auc":       round(roc_auc_score(y_true, y_prob), 5),
        "f1":        round(f1_score(y_true, yp, zero_division=0), 5),
        "precision": round(precision_score(y_true, yp, zero_division=0), 5),
        "recall":    round(recall_score(y_true, yp, zero_division=0), 5),
        "accuracy":  round(accuracy_score(y_true, yp), 5),
    }


def reg_metrics(y_true, y_pred) -> dict:
    return {
        "mae":  round(mean_absolute_error(y_true, y_pred), 4),
        "rmse": round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
    }


def build_lstm_arrays(
    df_pl: pl.DataFrame,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Build (N, 3, F) step matrix for a single df slice."""
    pdf = df_pl.to_pandas().copy()
    for c in LSTM_NUMERIC_FILL:
        if c in pdf.columns:
            pdf[c] = pdf[c].fillna(pdf[c].median())

    n = len(pdf)
    F = len(STEP_FEATURES)
    X = np.zeros((n, 3, F), dtype=np.float32)

    def si(name):  # step-feature index
        return STEP_FEATURES.index(name)

    # prev2 step
    X[:, 0, si("arr_delay_prev")] = pdf["prev2_arr_delay"].values
    X[:, 0, si("dep_delay_prev")] = pdf["prev2_dep_delay"].values
    X[:, 0, si("arr_del15_prev")] = pdf["prev2_arr_del15"].values
    X[:, 0, si("dep_del15_prev")] = pdf["prev2_dep_del15"].values
    X[:, 0, si("gap_minutes")]    = pdf["time_since_prev2_arrival_minutes"].values

    # prev1 step
    X[:, 1, si("arr_delay_prev")] = pdf["prev1_arr_delay"].values
    X[:, 1, si("dep_delay_prev")] = pdf["prev1_dep_delay"].values
    X[:, 1, si("arr_del15_prev")] = pdf["prev1_arr_del15"].values
    X[:, 1, si("dep_del15_prev")] = pdf["prev1_dep_del15"].values
    X[:, 1, si("gap_minutes")]    = pdf["prev1_turnaround_minutes"].values

    # current step
    for col in LSTM_CURRENT_FEATURES:
        if col in STEP_FEATURES:
            X[:, 2, si(col)] = pdf[col].values

    y_c = pdf[TARGET_CLASS].astype(int).values
    y_r = pdf[TARGET_REG].astype(float).values
    return X, y_c, y_r


def scale_lstm(
    X_tr: np.ndarray, X_va: np.ndarray
) -> tuple[np.ndarray, np.ndarray, StandardScaler]:
    nt, ts, nf = X_tr.shape
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr.reshape(-1, nf)).reshape(nt, ts, nf).astype(np.float32)
    X_va_s = sc.transform(X_va.reshape(-1, nf)).reshape(X_va.shape[0], ts, nf).astype(np.float32)
    return X_tr_s, X_va_s, sc


def build_lstm_model(input_shape, units1, units2, dropout, lr=LSTM_LR) -> tf.keras.Model:
    inp     = Input(shape=input_shape)
    x       = LSTM(units1, return_sequences=True)(inp)
    x       = Dropout(dropout)(x)
    x       = LSTM(units2)(x)
    x       = Dropout(dropout)(x)
    cls_out = Dense(1, activation="sigmoid", name="cls")(x)
    reg_out = Dense(1, activation="linear",  name="reg")(x)
    model   = Model(inputs=inp, outputs=[cls_out, reg_out])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss    = {"cls": "binary_crossentropy", "reg": "mse"},
        metrics = {"cls": [tf.keras.metrics.AUC(name="auc")], "reg": ["mae"]},
    )
    return model


# ============================================================
# 6. XGBoost GRID SEARCH  (on Fold 1)
# ============================================================

section("6. XGBoost GRID SEARCH  (Fold 1 only)")

# Use n_estimators=500 with early stopping — more principled than sweeping it
XGB_N_ESTIMATORS   = 500
XGB_EARLY_STOPPING = 30   # rounds without improvement on eval set

def _xgb_fit_one(params, X_tr, y_tr, X_va, y_va, task="cls"):
    """Train one XGB model with early stopping on a validation slice."""
    xgb_device = "cuda" if USE_GPU else "cpu"
    common = dict(**params, n_jobs=-1, device=xgb_device, random_state=42,
                  n_estimators=XGB_N_ESTIMATORS, early_stopping_rounds=XGB_EARLY_STOPPING)

    if task == "cls":
        m = XGBClassifier(**common, eval_metric="logloss")
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        prob = m.predict_proba(X_va)[:, 1]
        return m, prob
    else:
        m = XGBRegressor(**common, eval_metric="rmse")
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        pred = m.predict(X_va)
        return m, pred


def xgb_grid_search_fold1(flights: pl.DataFrame) -> tuple[dict, pd.DataFrame]:
    """Run XGB grid search on Fold 1, return best params + full results table."""
    label, train_end, val_end = CV_FOLDS[0]
    train_df, val_df = fold_split(flights, train_end, val_end)

    print(f"  Grid-search fold: '{label}'  "
          f"train={train_df.height:,}  val={val_df.height:,}")

    X_tr = safe_fill(train_df, XGB_FEATURES)
    X_va = safe_fill(val_df,   XGB_FEATURES)
    y_tr_c = train_df[TARGET_CLASS].to_pandas().astype(int)
    y_va_c = val_df[TARGET_CLASS].to_pandas().astype(int)
    y_tr_r = train_df[TARGET_REG].to_pandas().astype(float)
    y_va_r = val_df[TARGET_REG].to_pandas().astype(float)

    keys   = list(XGB_PARAM_GRID.keys())
    combos = list(itertools.product(*[XGB_PARAM_GRID[k] for k in keys]))
    total  = len(combos)
    print(f"  {total} combos to evaluate …\n")

    rows     = []
    best_auc = -np.inf
    best_p   = {}

    for i, vals in enumerate(combos, 1):
        params = dict(zip(keys, vals))
        t_c = time.perf_counter()

        clf, y_prob = _xgb_fit_one(params, X_tr, y_tr_c, X_va, y_va_c, "cls")
        reg, y_pred = _xgb_fit_one(params, X_tr, y_tr_r, X_va, y_va_r, "reg")

        cm = cls_metrics(y_va_c, y_prob)
        rm = reg_metrics(y_va_r, y_pred)

        row = {"combo": i}
        row.update(params)
        row.update({f"cls_{k}": v for k, v in cm.items()})
        row.update({f"reg_{k}": v for k, v in rm.items()})
        rows.append(row)

        elapsed = time.perf_counter() - t_c
        print(f"  [{i:>2}/{total}] AUC={cm['auc']:.4f}  MAE={rm['mae']:.2f}  "
              f"params={params}  ({elapsed:.1f}s)")

        if cm["auc"] > best_auc:
            best_auc = cm["auc"]
            best_p   = params.copy()

    df = pd.DataFrame(rows).sort_values("cls_auc", ascending=False).reset_index(drop=True)
    print(f"\n  ✓ XGB best AUC={best_auc:.4f}  params={best_p}")
    return best_p, df


t0 = time.perf_counter()
xgb_best_params, xgb_gs_df = xgb_grid_search_fold1(flights)
timer_log("XGBoost grid search", t0)


# ============================================================
# 7. LSTM GRID SEARCH  (on Fold 1)
# ============================================================

section("7. LSTM GRID SEARCH  (Fold 1 only)")

def lstm_grid_search_fold1(flights: pl.DataFrame) -> tuple[dict, pd.DataFrame]:
    """Run LSTM grid search on Fold 1, return best params + full results table."""
    label, train_end, val_end = CV_FOLDS[0]
    train_df, val_df = fold_split(flights, train_end, val_end)

    print(f"  Grid-search fold: '{label}'  "
          f"train={train_df.height:,}  val={val_df.height:,}")

    # Build arrays once; they are reused across all combos
    X_tr_raw, y_tr_c, y_tr_r = build_lstm_arrays(train_df)
    X_va_raw, y_va_c, y_va_r = build_lstm_arrays(val_df)
    X_tr_s, X_va_s, _        = scale_lstm(X_tr_raw, X_va_raw)

    y_tr_c = y_tr_c.astype(np.float32);  y_tr_r = y_tr_r.astype(np.float32)
    y_va_c = y_va_c.astype(np.float32);  y_va_r = y_va_r.astype(np.float32)
    input_shape = (X_tr_s.shape[1], X_tr_s.shape[2])

    keys   = list(LSTM_PARAM_GRID.keys())
    combos = list(itertools.product(*[LSTM_PARAM_GRID[k] for k in keys]))
    total  = len(combos)
    print(f"  {total} combos to evaluate …\n")

    rows     = []
    best_auc = -np.inf
    best_p   = {}

    for i, vals in enumerate(combos, 1):
        params = dict(zip(keys, vals))
        t_c = time.perf_counter()

        tf.keras.backend.clear_session()
        model = build_lstm_model(input_shape, **params)

        model.fit(
            X_tr_s,
            {"cls": y_tr_c, "reg": y_tr_r},
            validation_split = LSTM_VAL_SPLIT,
            epochs           = LSTM_EPOCHS,
            batch_size       = LSTM_BATCH_SIZE,
            callbacks        = [EarlyStopping(monitor="val_loss", patience=3,
                                              restore_best_weights=True)],
            verbose=0,
        )

        pc, pr = model.predict(X_va_s, verbose=0)
        pc = pc.ravel();  pr = pr.ravel()

        cm = cls_metrics(y_va_c.astype(int), pc)
        rm = reg_metrics(y_va_r, pr)

        row = {"combo": i}
        row.update(params)
        row.update({f"cls_{k}": v for k, v in cm.items()})
        row.update({f"reg_{k}": v for k, v in rm.items()})
        rows.append(row)

        elapsed = time.perf_counter() - t_c
        print(f"  [{i:>2}/{total}] AUC={cm['auc']:.4f}  MAE={rm['mae']:.2f}  "
              f"params={params}  ({elapsed:.1f}s)")

        if cm["auc"] > best_auc:
            best_auc = cm["auc"]
            best_p   = params.copy()

    df = pd.DataFrame(rows).sort_values("cls_auc", ascending=False).reset_index(drop=True)
    print(f"\n  ✓ LSTM best AUC={best_auc:.4f}  params={best_p}")
    return best_p, df


t0 = time.perf_counter()
lstm_best_params, lstm_gs_df = lstm_grid_search_fold1(flights)
timer_log("LSTM grid search", t0)


# ============================================================
# 8. WALK-FORWARD CV WITH BEST PARAMS
# ============================================================

section("8. WALK-FORWARD CV  (best params across all folds)")

def xgb_cv(flights: pl.DataFrame, params: dict) -> pd.DataFrame:
    """Evaluate best XGB params on every CV fold."""
    rows = []
    for label, train_end, val_end in CV_FOLDS:
        t_f = time.perf_counter()
        train_df, val_df = fold_split(flights, train_end, val_end)

        X_tr = safe_fill(train_df, XGB_FEATURES)
        X_va = safe_fill(val_df,   XGB_FEATURES)
        y_tr_c = train_df[TARGET_CLASS].to_pandas().astype(int)
        y_va_c = val_df[TARGET_CLASS].to_pandas().astype(int)
        y_tr_r = train_df[TARGET_REG].to_pandas().astype(float)
        y_va_r = val_df[TARGET_REG].to_pandas().astype(float)

        _, y_prob = _xgb_fit_one(params, X_tr, y_tr_c, X_va, y_va_c, "cls")
        _, y_pred = _xgb_fit_one(params, X_tr, y_tr_r, X_va, y_va_r, "reg")

        row = {"fold": label, "train_rows": train_df.height, "val_rows": val_df.height}
        row.update({f"cls_{k}": v for k, v in cls_metrics(y_va_c, y_prob).items()})
        row.update({f"reg_{k}": v for k, v in reg_metrics(y_va_r, y_pred).items()})
        rows.append(row)

        timer_log(f"  XGB {label}", t_f)

    return pd.DataFrame(rows)


def lstm_cv(flights: pl.DataFrame, params: dict) -> pd.DataFrame:
    """Evaluate best LSTM params on every CV fold."""
    rows = []
    for label, train_end, val_end in CV_FOLDS:
        t_f = time.perf_counter()
        train_df, val_df = fold_split(flights, train_end, val_end)

        X_tr_raw, y_tr_c, y_tr_r = build_lstm_arrays(train_df)
        X_va_raw, y_va_c, y_va_r = build_lstm_arrays(val_df)
        X_tr_s, X_va_s, _        = scale_lstm(X_tr_raw, X_va_raw)

        y_tr_c_f = y_tr_c.astype(np.float32);  y_tr_r_f = y_tr_r.astype(np.float32)
        input_shape = (X_tr_s.shape[1], X_tr_s.shape[2])

        tf.keras.backend.clear_session()
        model = build_lstm_model(input_shape, **params)
        model.fit(
            X_tr_s,
            {"cls": y_tr_c_f, "reg": y_tr_r_f},
            validation_split = LSTM_VAL_SPLIT,
            epochs           = LSTM_EPOCHS,
            batch_size       = LSTM_BATCH_SIZE,
            callbacks        = [EarlyStopping(monitor="val_loss", patience=3,
                                              restore_best_weights=True)],
            verbose=0,
        )

        pc, pr = model.predict(X_va_s, verbose=0)
        pc = pc.ravel();  pr = pr.ravel()

        row = {"fold": label, "train_rows": train_df.height, "val_rows": val_df.height}
        row.update({f"cls_{k}": v for k, v in cls_metrics(y_va_c, pc).items()})
        row.update({f"reg_{k}": v for k, v in reg_metrics(y_va_r, pr).items()})
        rows.append(row)

        timer_log(f"  LSTM {label}", t_f)

    return pd.DataFrame(rows)


print("\n--- XGBoost CV ---")
t0 = time.perf_counter()
xgb_cv_df = xgb_cv(flights, xgb_best_params)
timer_log("XGBoost CV total", t0)

print("\n--- LSTM CV ---")
t0 = time.perf_counter()
lstm_cv_df = lstm_cv(flights, lstm_best_params)
timer_log("LSTM CV total", t0)

print("\n── XGBoost Walk-Forward CV Results ──")
print(xgb_cv_df[["fold","train_rows","val_rows","cls_auc","cls_f1","reg_mae","reg_rmse"]].to_string(index=False))

print("\n── LSTM Walk-Forward CV Results ──")
print(lstm_cv_df[["fold","train_rows","val_rows","cls_auc","cls_f1","reg_mae","reg_rmse"]].to_string(index=False))


# ============================================================
# 9. FINAL MODELS — retrain on Jan–Oct, evaluate on Nov–Dec
#    (Fold 3 = the canonical holdout)
# ============================================================

section("9. FINAL MODELS — Jan–Oct train / Nov–Dec holdout")

_, train_end_final, val_end_final = CV_FOLDS[-1]   # Fold 3 boundaries
train_final, test_final = fold_split(flights, train_end_final, val_end_final)
print(f"  Final train: {train_final.height:,}  |  Final test: {test_final.height:,}")

# ---- XGBoost ----
t0 = time.perf_counter()

X_tr_f  = safe_fill(train_final, XGB_FEATURES)
X_te_f  = safe_fill(test_final,  XGB_FEATURES)
y_tr_c  = train_final[TARGET_CLASS].to_pandas().astype(int)
y_te_c  = test_final[TARGET_CLASS].to_pandas().astype(int)
y_tr_r  = train_final[TARGET_REG].to_pandas().astype(float)
y_te_r  = test_final[TARGET_REG].to_pandas().astype(float)

xgb_device = "cuda" if USE_GPU else "cpu"
final_xgb_clf = XGBClassifier(
    **xgb_best_params, n_estimators=XGB_N_ESTIMATORS,
    early_stopping_rounds=XGB_EARLY_STOPPING,
    eval_metric="logloss", n_jobs=-1, device=xgb_device, random_state=42,
)
# Use 10% of training as internal eval for early stopping
val_frac = int(len(X_tr_f) * 0.10)
final_xgb_clf.fit(
    X_tr_f.iloc[:-val_frac], y_tr_c.iloc[:-val_frac],
    eval_set=[(X_tr_f.iloc[-val_frac:], y_tr_c.iloc[-val_frac:])],
    verbose=False,
)
final_xgb_reg = XGBRegressor(
    **xgb_best_params, n_estimators=XGB_N_ESTIMATORS,
    early_stopping_rounds=XGB_EARLY_STOPPING,
    eval_metric="rmse", n_jobs=-1, device=xgb_device, random_state=42,
)
final_xgb_reg.fit(
    X_tr_f.iloc[:-val_frac], y_tr_r.iloc[:-val_frac],
    eval_set=[(X_tr_f.iloc[-val_frac:], y_tr_r.iloc[-val_frac:])],
    verbose=False,
)

xgb_final_prob = final_xgb_clf.predict_proba(X_te_f)[:, 1]
xgb_final_pred = final_xgb_reg.predict(X_te_f)
xgb_final_cls  = cls_metrics(y_te_c, xgb_final_prob)
xgb_final_reg  = reg_metrics(y_te_r, xgb_final_pred)

timer_log("XGB final train", t0)

# ---- LSTM ----
t0 = time.perf_counter()

X_tr_raw, y_tr_c_np, y_tr_r_np = build_lstm_arrays(train_final)
X_te_raw, y_te_c_np, y_te_r_np = build_lstm_arrays(test_final)
X_tr_s_f, X_te_s_f, _          = scale_lstm(X_tr_raw, X_te_raw)

y_tr_c_f = y_tr_c_np.astype(np.float32);  y_tr_r_f = y_tr_r_np.astype(np.float32)

tf.keras.backend.clear_session()
final_lstm = build_lstm_model(
    (X_tr_s_f.shape[1], X_tr_s_f.shape[2]), **lstm_best_params
)
final_lstm.fit(
    X_tr_s_f,
    {"cls": y_tr_c_f, "reg": y_tr_r_f},
    validation_split = LSTM_VAL_SPLIT,
    epochs           = LSTM_EPOCHS,
    batch_size       = LSTM_BATCH_SIZE,
    callbacks        = [EarlyStopping(monitor="val_loss", patience=3,
                                      restore_best_weights=True)],
    verbose=1,
)

lstm_pc, lstm_pr = final_lstm.predict(X_te_s_f, verbose=0)
lstm_pc = lstm_pc.ravel();  lstm_pr = lstm_pr.ravel()
lstm_final_cls = cls_metrics(y_te_c_np, lstm_pc)
lstm_final_reg = reg_metrics(y_te_r_np, lstm_pr)

timer_log("LSTM final train", t0)

# ---- Print final comparison ----
comparison_df = pd.DataFrame([
    {"model": "XGB  (tuned, Jan–Oct train)",
     **{f"cls_{k}": v for k, v in xgb_final_cls.items()},
     **{f"reg_{k}": v for k, v in xgb_final_reg.items()}},
    {"model": "LSTM (tuned, Jan–Oct train)",
     **{f"cls_{k}": v for k, v in lstm_final_cls.items()},
     **{f"reg_{k}": v for k, v in lstm_final_reg.items()}},
])
print("\n── Final Holdout Comparison (Nov–Dec) ─────────────────")
print(comparison_df.to_string(index=False))

print("\n── Best Hyperparameters ───────────────────────────────")
print("XGB :", json.dumps(xgb_best_params, indent=2))
print("LSTM:", json.dumps(lstm_best_params, indent=2))


# ============================================================
# 10. VISUALIZATIONS
# ============================================================

section("10. VISUALIZATIONS")

# ── Helper: concise bar-error chart ────────────────────────
def _bar_error(ax, labels, means, stds, title, ylabel, color):
    x = np.arange(len(labels))
    bars = ax.bar(x, means, yerr=stds, capsize=5, color=color, alpha=0.75, ecolor="black")
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=25, ha="right", fontsize=8)
    ax.set_title(title, fontsize=10); ax.set_ylabel(ylabel)
    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(stds)*0.05,
                f"{m:.3f}", ha="center", va="bottom", fontsize=7)


# ── 10a. Grid Search — AUC & MAE distributions ─────────────
def plot_gs_distributions(gs_df, model_label):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f"Grid Search — {model_label}", fontsize=12)
    for ax, col, best_fn, xlabel, marker_label in [
        (axes[0], "cls_auc", max, "AUC",  "best"),
        (axes[1], "reg_mae", min, "MAE",  "best"),
    ]:
        ax.hist(gs_df[col], bins=max(5, len(gs_df)//3), edgecolor="black", alpha=0.8)
        ax.axvline(best_fn(gs_df[col]), color="crimson", linestyle="--", label=marker_label)
        ax.set_xlabel(xlabel); ax.set_ylabel("Count"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

plot_gs_distributions(xgb_gs_df,  "XGB full propagation (Fold 1)")
plot_gs_distributions(lstm_gs_df, "LSTM context full (Fold 1)")


# ── 10b. Parameter sensitivity — box plots ─────────────────
def plot_param_sensitivity(gs_df, params, model_label):
    ncols = len(params)
    fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 4))
    if ncols == 1: axes = [axes]
    fig.suptitle(f"Param Sensitivity (AUC) — {model_label}", fontsize=11)
    for ax, p in zip(axes, params):
        vals  = sorted(gs_df[p].unique())
        data  = [gs_df[gs_df[p] == v]["cls_auc"].values for v in vals]
        ax.boxplot(data, labels=[str(v) for v in vals])
        ax.set_xlabel(p); ax.set_ylabel("AUC")
    plt.tight_layout(); plt.show()

plot_param_sensitivity(xgb_gs_df,  ["max_depth", "learning_rate", "subsample"],
                       "XGB full propagation")
plot_param_sensitivity(lstm_gs_df, ["units1", "units2", "dropout"],
                       "LSTM context full")


# ── 10c. Walk-Forward CV — AUC & MAE across folds ──────────
def plot_cv_stability(xgb_df, lstm_df):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    folds = xgb_df["fold"].tolist()
    x     = np.arange(len(folds))
    w     = 0.35

    # AUC
    axes[0].bar(x - w/2, xgb_df["cls_auc"],  w, label="XGB",  alpha=0.8)
    axes[0].bar(x + w/2, lstm_df["cls_auc"], w, label="LSTM", alpha=0.8)
    axes[0].set_xticks(x); axes[0].set_xticklabels(folds, rotation=20, ha="right", fontsize=8)
    axes[0].set_ylabel("AUC"); axes[0].set_title("AUC across CV Folds")
    axes[0].legend(); axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    # Add value labels
    for rect in axes[0].patches:
        h = rect.get_height()
        axes[0].text(rect.get_x() + rect.get_width()/2, h + 0.0005,
                     f"{h:.3f}", ha="center", va="bottom", fontsize=7)

    # MAE
    axes[1].bar(x - w/2, xgb_df["reg_mae"],  w, label="XGB",  alpha=0.8)
    axes[1].bar(x + w/2, lstm_df["reg_mae"], w, label="LSTM", alpha=0.8)
    axes[1].set_xticks(x); axes[1].set_xticklabels(folds, rotation=20, ha="right", fontsize=8)
    axes[1].set_ylabel("MAE (minutes)"); axes[1].set_title("MAE across CV Folds")
    axes[1].legend()
    for rect in axes[1].patches:
        h = rect.get_height()
        axes[1].text(rect.get_x() + rect.get_width()/2, h + 0.02,
                     f"{h:.2f}", ha="center", va="bottom", fontsize=7)

    plt.suptitle("Walk-Forward CV Stability", fontsize=13)
    plt.tight_layout(); plt.show()

plot_cv_stability(xgb_cv_df, lstm_cv_df)


# ── 10d. CV variance summary table ─────────────────────────
def cv_summary_table(xgb_df, lstm_df):
    rows = []
    for name, df in [("XGB (tuned)", xgb_df), ("LSTM (tuned)", lstm_df)]:
        rows.append({
            "model":    name,
            "auc_mean": df["cls_auc"].mean(),
            "auc_std":  df["cls_auc"].std(),
            "f1_mean":  df["cls_f1"].mean(),
            "f1_std":   df["cls_f1"].std(),
            "mae_mean": df["reg_mae"].mean(),
            "mae_std":  df["reg_mae"].std(),
            "rmse_mean":df["reg_rmse"].mean(),
            "rmse_std": df["reg_rmse"].std(),
        })
    return pd.DataFrame(rows)

cv_summary = cv_summary_table(xgb_cv_df, lstm_cv_df)
print("\n── CV Summary (mean ± std across 3 folds) ────────────")
print(cv_summary.round(4).to_string(index=False))


# ── 10e. ROC Curves — final holdout ────────────────────────
plt.figure(figsize=(8, 6))
for label, y_prob in [
    ("XGB  (tuned)", xgb_final_prob),
    ("LSTM (tuned)", lstm_pc),
]:
    fpr, tpr, _ = roc_curve(y_te_c, y_prob)
    auc_val      = roc_auc_score(y_te_c, y_prob)
    plt.plot(fpr, tpr, label=f"{label}  AUC={auc_val:.4f}", linewidth=1.8)
plt.plot([0, 1], [0, 1], "k--", linewidth=1)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Final Holdout (Nov–Dec)")
plt.legend(); plt.tight_layout(); plt.show()


# ── 10f. Actual vs Predicted scatter (both models) ─────────
def plot_avp(y_true, y_pred, label, n=5_000):
    rng = np.random.RandomState(42)
    idx = rng.choice(len(y_true), size=min(n, len(y_true)), replace=False)
    plt.figure(figsize=(7, 6))
    plt.scatter(np.array(y_true)[idx], np.array(y_pred)[idx], alpha=0.25, s=8)
    lo = min(np.array(y_true)[idx].min(), np.array(y_pred)[idx].min())
    hi = max(np.array(y_true)[idx].max(), np.array(y_pred)[idx].max())
    plt.plot([lo, hi], [lo, hi], "r--", linewidth=1, label="perfect fit")
    plt.xlabel("Actual Delay (min)"); plt.ylabel("Predicted Delay (min)")
    plt.title(f"Actual vs Predicted — {label}")
    plt.legend(); plt.tight_layout(); plt.show()

plot_avp(y_te_r, xgb_final_pred, "XGB (tuned, Nov–Dec)")
plot_avp(y_te_r, lstm_pr,        "LSTM (tuned, Nov–Dec)")


# ── 10g. XGBoost feature importance ────────────────────────
imp_df = (
    pd.DataFrame({"feature": XGB_FEATURES,
                  "importance": final_xgb_clf.feature_importances_})
    .sort_values("importance", ascending=False)
    .head(20)
)
plt.figure(figsize=(10, 7))
plt.barh(imp_df["feature"][::-1], imp_df["importance"][::-1])
plt.title("Top-20 Feature Importances — Final XGB (Nov–Dec holdout)")
plt.tight_layout(); plt.show()


# ── 10h. AUC trend line across folds ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fold_labels = [f.replace("Fold", "F").replace(" val ", "\n") for f in xgb_cv_df["fold"]]

for ax, col, ylabel, title in [
    (axes[0], "cls_auc",  "AUC",           "AUC trend across folds"),
    (axes[1], "reg_mae",  "MAE (minutes)",  "MAE trend across folds"),
]:
    ax.plot(fold_labels, xgb_cv_df[col],  marker="o", label="XGB",  linewidth=1.8)
    ax.plot(fold_labels, lstm_cv_df[col], marker="s", label="LSTM", linewidth=1.8)
    ax.set_ylabel(ylabel); ax.set_title(title); ax.legend()
    ax.tick_params(axis="x", labelsize=8)

plt.suptitle("Seasonal Stability — Walk-Forward CV", fontsize=12)
plt.tight_layout(); plt.show()


# ============================================================
# 11. TOP COMBOS SUMMARY TABLES
# ============================================================

section("11. TOP COMBOS SUMMARY")

xgb_cols  = ["combo"] + list(XGB_PARAM_GRID.keys()) + ["cls_auc","cls_f1","reg_mae","reg_rmse"]
lstm_cols = ["combo"] + list(LSTM_PARAM_GRID.keys()) + ["cls_auc","cls_f1","reg_mae","reg_rmse"]

print("\n── XGB Top 10 Combos (Fold 1) ─────────────────────────")
print(xgb_gs_df.head(10)[xgb_cols].to_string(index=False))

print("\n── LSTM Top 10 Combos (Fold 1) ────────────────────────")
print(lstm_gs_df.head(10)[lstm_cols].to_string(index=False))

print("\n── CV Fold Detail — XGB ───────────────────────────────")
print(xgb_cv_df.to_string(index=False))

print("\n── CV Fold Detail — LSTM ──────────────────────────────")
print(lstm_cv_df.to_string(index=False))

print("\n── CV Summary (mean ± std) ────────────────────────────")
print(cv_summary.round(4).to_string(index=False))

print("\n── Final Holdout (Nov–Dec) ────────────────────────────")
print(comparison_df.to_string(index=False))

print()
timer_log("TOTAL PIPELINE TIME", GLOBAL_START)